In [1]:
import time
from pyspark.sql.functions import current_timestamp, date_format

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline

from feature_store_pyspark.FeatureStoreManager import FeatureStoreManager

# Initialize spark

In [2]:
spark = (SparkSession.builder
         .master("local[*]")
         .appName("EMR-Local-Dev")
         .getOrCreate())

# This will now work without the InvalidClassException
print(spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


3.5.2-amzn-1


# Load data

In [3]:

df_schema = StructType([
    StructField('Customer_ID', StringType(), True),
    StructField('Age', IntegerType(), True),
    StructField('Gender', StringType(), True),
    StructField('Annual_Income', IntegerType(), True),
    StructField('Spending_score', IntegerType(), True),
    StructField('Region', StringType(), True),
    StructField('Marital_Status', StringType(), True),
    StructField('Num_of_Children', IntegerType(), True),
    StructField('Employment_Status', StringType(), True),
    StructField('Credit_Score', IntegerType(), True),
    StructField('Online_Shopping_Frequency', StringType(), True),
    StructField('Target', StringType(), True)
])

df = spark.read.csv('/home/hadoop/test_data/Customer_Data.csv', header=True, schema=df_schema)
df.show(5)

+-----------+---+------+-------------+--------------+------+--------------+---------------+-----------------+------------+-------------------------+------+
|Customer_ID|Age|Gender|Annual_Income|Spending_score|Region|Marital_Status|Num_of_Children|Employment_Status|Credit_Score|Online_Shopping_Frequency|Target|
+-----------+---+------+-------------+--------------+------+--------------+---------------+-----------------+------------+-------------------------+------+
|     CUST_1| 25|Female|        66041|            15| South|       Widowed|              1|       Unemployed|         332|                       12|     0|
|     CUST_2| 66|  Male|        46937|            39| South|        Single|              0|          Retired|         550|                        4|     0|
|     CUST_3| 18|  Male|       142885|            65| South|        Single|              2|          Retired|         503|                        7|     0|
|     CUST_4| 19|Female|       146383|            91| South|    

# Understand the data

In [4]:
# Find target counts
df.filter(df['Target'] == 1).count(), df.filter(df['Target'] == 0).count()

(9136, 20864)

In [5]:
# Distinct employment status
df.select('Employment_Status').distinct().show()

+-----------------+
|Employment_Status|
+-----------------+
|       Unemployed|
|          Retired|
|         Employed|
|          Student|
+-----------------+



# Transform

In [6]:
cat_features = ['Gender', 'Region', 'Marital_Status', 'Employment_Status', 'Online_Shopping_Frequency']

# Basic impute
df = df.na.fill(value=0.0)
df = df.na.fill(value="N/A")

# Add creation time
df = df.withColumn('created_time', date_format(current_timestamp(), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))

# Define encoders
indexers = [
    StringIndexer(inputCol = c, outputCol = c + "_tmp", handleInvalid = "keep")
    for c in cat_features
]
encoders = [
    OneHotEncoder(inputCol = c + "_tmp", outputCol = c + "_vec")
    for c in cat_features
]

# Encode
pipeline = Pipeline(stages = indexers + encoders)
model = pipeline.fit(df)
encoded_df = model.transform(df)
df.show(5, truncate=False)


+-----------+---+------+-------------+--------------+------+--------------+---------------+-----------------+------------+-------------------------+------+------------------------+
|Customer_ID|Age|Gender|Annual_Income|Spending_score|Region|Marital_Status|Num_of_Children|Employment_Status|Credit_Score|Online_Shopping_Frequency|Target|created_time            |
+-----------+---+------+-------------+--------------+------+--------------+---------------+-----------------+------------+-------------------------+------+------------------------+
|CUST_1     |25 |Female|66041        |15            |South |Widowed       |1              |Unemployed       |332         |12                       |0     |2026-03-03T03:48:06.875Z|
|CUST_2     |66 |Male  |46937        |39            |South |Single        |0              |Retired          |550         |4                        |0     |2026-03-03T03:48:06.875Z|
|CUST_3     |18 |Male  |142885       |65            |South |Single        |2              |Reti

In [7]:

# Drop unneeded columns
for c in cat_features:
    encoded_df = (encoded_df
                      .drop(c)
                      .drop(f'{c}_tmp')
                      .withColumnRenamed(f'{c}_vec', c)
                  )

In [8]:
encoded_df.show(truncate=False)

+-----------+---+-------------+--------------+---------------+------------+------+------------------------+-------------+-------------+--------------+-----------------+-------------------------+
|Customer_ID|Age|Annual_Income|Spending_score|Num_of_Children|Credit_Score|Target|created_time            |Gender       |Region       |Marital_Status|Employment_Status|Online_Shopping_Frequency|
+-----------+---+-------------+--------------+---------------+------------+------+------------------------+-------------+-------------+--------------+-----------------+-------------------------+
|CUST_1     |25 |66041        |15            |1              |332         |0     |2026-03-03T03:48:13.643Z|(2,[1],[1.0])|(4,[3],[1.0])|(4,[0],[1.0]) |(4,[0],[1.0])    |(20,[16],[1.0])          |
|CUST_2     |66 |46937        |39            |0              |550         |0     |2026-03-03T03:48:13.643Z|(2,[0],[1.0])|(4,[3],[1.0])|(4,[1],[1.0]) |(4,[3],[1.0])    |(20,[3],[1.0])           |
|CUST_3     |18 |142885  

# Flatten vectors

In [9]:
feature_labels = {}
for i, c in enumerate(cat_features):
    labels = model.stages[i].labels
    feature_labels[c] = labels[:-1]

feature_labels

{'Gender': ['Male'],
 'Region': ['North', 'East', 'West'],
 'Marital_Status': ['Widowed', 'Single', 'Married'],
 'Employment_Status': ['Unemployed', 'Student', 'Employed'],
 'Online_Shopping_Frequency': ['18',
  '19',
  '5',
  '4',
  '17',
  '0',
  '7',
  '14',
  '15',
  '16',
  '3',
  '9',
  '13',
  '10',
  '1',
  '2',
  '12',
  '6',
  '11']}

In [10]:
for c in cat_features:
    labels = feature_labels[c]
    for i, label in enumerate(labels):
        col_name = f"{c}_{label}".replace(" ", "_")
        encoded_df = encoded_df.withColumn(col_name, F.udf(lambda v: int(v[i]), "int")(F.col(c)))

    encoded_df = encoded_df.drop(c)


encoded_df.show(5)

26/03/03 03:48:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 27:>                                                         (0 + 1) / 1]

+-----------+---+-------------+--------------+---------------+------------+------+--------------------+-----------+------------+-----------+-----------+----------------------+---------------------+----------------------+----------------------------+-------------------------+--------------------------+----------------------------+----------------------------+---------------------------+---------------------------+----------------------------+---------------------------+---------------------------+----------------------------+----------------------------+----------------------------+---------------------------+---------------------------+----------------------------+----------------------------+---------------------------+---------------------------+----------------------------+---------------------------+----------------------------+
|Customer_ID|Age|Annual_Income|Spending_score|Num_of_Children|Credit_Score|Target|        created_time|Gender_Male|Region_North|Region_East|Region_West|Mar

In [11]:
encoded_df.columns

['Customer_ID',
 'Age',
 'Annual_Income',
 'Spending_score',
 'Num_of_Children',
 'Credit_Score',
 'Target',
 'created_time',
 'Gender_Male',
 'Region_North',
 'Region_East',
 'Region_West',
 'Marital_Status_Widowed',
 'Marital_Status_Single',
 'Marital_Status_Married',
 'Employment_Status_Unemployed',
 'Employment_Status_Student',
 'Employment_Status_Employed',
 'Online_Shopping_Frequency_18',
 'Online_Shopping_Frequency_19',
 'Online_Shopping_Frequency_5',
 'Online_Shopping_Frequency_4',
 'Online_Shopping_Frequency_17',
 'Online_Shopping_Frequency_0',
 'Online_Shopping_Frequency_7',
 'Online_Shopping_Frequency_14',
 'Online_Shopping_Frequency_15',
 'Online_Shopping_Frequency_16',
 'Online_Shopping_Frequency_3',
 'Online_Shopping_Frequency_9',
 'Online_Shopping_Frequency_13',
 'Online_Shopping_Frequency_10',
 'Online_Shopping_Frequency_1',
 'Online_Shopping_Frequency_2',
 'Online_Shopping_Frequency_12',
 'Online_Shopping_Frequency_6',
 'Online_Shopping_Frequency_11']